In [ ]:

try:
    import kaggle_benchmarks as kbench
except ImportError:
    # Failsafe: Mock the kbench harness if the library is not found (e.g. during build/commit)
    import types
    class MockTask:
        def __init__(self, f): self.f = f
        def run(self, *args, **kwargs): return self.f(*args, **kwargs)
        def evaluate(self, *args, **kwargs):
            class Res: 
                def as_dataframe(self): return None
            return Res()
        def __call__(self, *args, **kwargs): return self.f(*args, **kwargs)
    kbench = types.SimpleNamespace()
    kbench.task = lambda **kwargs: lambda f: MockTask(f)
    kbench.llm = types.SimpleNamespace(prompt=lambda p: '{"final_answer": "0.0"}')
    print('⚠️ kaggle_benchmarks not found. Running in Failsafe (Mock) mode.')

import json
import re
import math
from datetime import datetime

# (Rest of the utils remain the same...)
def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence: blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end + 1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, ground_truth, rel_tol=0.015):
    def parse_physics_number(text):
        s = str(text).replace(",", "").strip().lower()
        s = re.sub(r"\\times\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        s = re.sub(r"\*\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if match:
            try: return float(match.group(0).replace("^", "e"))
            except: return None
        return None
    pred = parse_physics_number(answer_text)
    try:
        target = float(ground_truth)
        if pred is None: return False
        if target == 0: return abs(pred) < 1e-9
        return math.isclose(pred, target, rel_tol=rel_tol)
    except: return False

# Task 15: Rotating Wire Action
TASK_ID = "fp_15"
GROUND_TRUTH = 22.392

@kbench.task(name="FP-15 Rotating Wire Action", description="Physics")
def task_15(llm) -> tuple[int, int]:
    prompt = """You are solving a frontier physics problem. Return valid JSON only.\n\nA rigid straight wire is welded to a rotor that spins with constant angular velocity Omega=3 rad/s about a fixed axis A (not vertical). The axis A lies in the x-z plane and is tilted by beta=30 degrees away from the vertical Z axis. The closest-approach distance between the wire and the axis A is A=1m at a point P. At P, the wire makes an angle theta=60 degrees with the horizontal plane, and the wire is oriented so that moving upward along the wire has a horizontal component in the direction of the rotor’s motion at P. A particle of mass m=2kg moves along the wire with a constant relative speed s_dot=2m/s. What is the action S = integral from 0 to 1 of L dt, expressed to 3 decimal places in Joule-seconds?\n\nReturn JSON: {\"final_answer\": \"<value>\"}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    final_ans = parsed.get("final_answer", "") if parsed else ""
    passed = numeric_pass(final_ans, GROUND_TRUTH)
    return (1 if passed else 0, 1)


In [ ]:
task_15.run(kbench.llm)
